# 5 - Transfer Learning (EfficientNetB0)Leverage ImageNet-pretrained features for high accuracy with minimal overfitting.**Phase 1**: Train with frozen backbone (only classification head learns).**Phase 2**: Unfreeze top layers and fine-tune at a lower learning rate.

In [ ]:
import sysfrom pathlib import Pathsys.path.insert(0, str(Path.cwd().parent))from src.config import Configfrom src.utils import set_global_seedfrom src.data_loader import create_all_pipelines, compute_class_weightsfrom src.models.transfer_learning import build_transfer_model, unfreeze_backbonefrom src.training import train_model, compile_modelfrom src.evaluation import evaluate_modelcfg = Config()cfg.epochs = 30cfg.initial_lr = 1e-3set_global_seed(cfg.seed)train_ds, val_ds, test_ds = create_all_pipelines(cfg, augment_train=True)class_weights = compute_class_weights(cfg)

## Phase 1 - Frozen Backbone Training

In [ ]:
model = build_transfer_model(cfg, freeze_backbone=True, augment=True)model.summary()

In [ ]:
history_phase1 = train_model(    model,    train_ds,    val_ds,    cfg,    experiment_name='transfer_phase1',    class_weight=class_weights,)

## Phase 2 - Fine-Tuning

In [ ]:
# Unfreeze top 20 layers of the backboneunfreeze_backbone(model, num_layers_to_unfreeze=20)# Recompile with a lower learning rate for fine-tuningcfg.epochs = 20cfg.initial_lr = 1e-4compile_model(model, cfg, label_smoothing=0.1)history_phase2 = train_model(    model,    train_ds,    val_ds,    cfg,    experiment_name='transfer_phase2',    class_weight=class_weights,)

In [ ]:
from src.utils import plot_training_historyplot_training_history(    cfg.experiment_dir('transfer_phase2') / 'history.csv',    title='Transfer Learning - Fine-Tuning',)

In [ ]:
loss, acc, top5 = evaluate_model(model, test_ds)